In [180]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import os
from dotenv import load_dotenv

load_dotenv()

True

In [181]:
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)


# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [182]:
query = """
SELECT DISTINCT v.Pedido,
    v.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2026-09-30'
    AND v.Fecha <= '2026-09-30'
    AND (Canal = 'Shopify' OR Ubicacion = 'Social Selling')
    AND v.Venta_Neta > 0
    AND v.Cliente NOT IN ('C22222222222')
GROUP BY  v.Pedido,
    v.Cliente,
    c.Nombres + ' ' + c.Apellidos,
    c.Email,
    c.Celular,
    v.Fecha,
    v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query, engine)
df_ventas.head()

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,104112,C32646403,NIDAL ELNESSER IBRAHIM,charif6@hotmail.com,3022685781,2026-09-30,CHATCENTER WEB,424981.72
1,104114,C43052395,CATHY DAVIES,cathydavies0601@gmail.com,3148218310,2026-09-30,Shopify,203352.60
2,104116,C45692165,CLAUDIA HELENA GUAPI,claudiaguapi78@gmail.com,3008102563,2026-09-30,CHATCENTER WEB,173311.88
3,104117,C1076331522,None,karicastao@gmail.com,3147657395,2026-09-30,Shopify,405864.90
4,104118,C1018419250,ZARETH GALVAN,zcgalvan@gmail.com,3243670638,2026-09-30,Shopify,190958.17


In [183]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Pedido,
# 	v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta,
#     Canal,
#     SUM(Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'AGOSTO_2026')
# 	AND Fecha BETWEEN '2026-09-25' AND '2026-09-30'
#     AND v.Cliente NOT IN ('C22222222222')
# GROUP BY  v.Pedido,
# 	v.Cliente,
#     cc.Nombres + ' ' +  cc.Apellidos,
#     cc.Email,
#     cc.Celular,
#     v.Fecha,
#     v.Canal
# """


# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [184]:
# Leer archivo
df_indigitall = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_indigitall[['phone', 'campaignName', 'sentAt',  'clicks']].copy()

# Renombrar columnas
df_campania.rename(columns={
    'phone': 'Celular',
    'campaignName': 'campania',
    'sentAt': 'fecha_envio',
    'clicks': 'clics'
}, inplace=True)

# Extraer primera fecha de apertura (si viene como lista en string)
df_campania['clics'] = df_campania['clics'].apply(
    lambda x: eval(x)[0] if isinstance(x, str) and x.startswith('[') and len(eval(x)) > 0 else None
)
df_campania['clics'] = pd.to_datetime(df_campania['clics'], errors='coerce').dt.tz_localize(None)
df_campania['fecha_envio'] = pd.to_datetime(df_campania['fecha_envio'], errors='coerce').dt.tz_localize(None).dt.date

df_enviados = df_campania[df_campania['fecha_envio'].notnull()].copy()
df_clics =  df_campania[df_campania['clics'].notnull()].copy()

fecha_maxima = pd.to_datetime('2026-09-30').date()
df_clics_filtrados = df_clics[df_clics['clics'].dt.date <= fecha_maxima]

df_clics_filtrados = df_clics_filtrados.drop_duplicates(subset='Celular', keep='first')

total_clics = df_clics_filtrados['Celular'].nunique()


In [185]:
# df_enviados['Celular'] = df_enviados['Celular'].astype(str)
# df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# # Merge por email
# df_merge = pd.merge(df_ventas, df_enviados, on='Celular', how='inner')

# df_merge

In [186]:
df_enviados['Celular'] = df_enviados['Celular'].astype(str)
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_merge = pd.merge(df_ventas, df_enviados, on='Celular', how='inner')

# Filtrar ventas dentro de 5 días desde la fecha de envio
df_merge['Fecha_Venta'] = pd.to_datetime(df_merge['Fecha_Venta'])
df_merge['fecha_envio'] = pd.to_datetime(df_merge['fecha_envio'])
df_merge['dias_post_apertura'] = (df_merge['Fecha_Venta'] - df_merge['fecha_envio']).dt.days
df_atribucion = df_merge[
    (df_merge['dias_post_apertura'] >= 0) &
    (df_merge['dias_post_apertura'] <= 5)
]

# Sumar la venta total atribuida
tottal_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Número de clientes que hicieron clic hasta {fecha_maxima}: {total_clics}")
print(f"Clientes Efectivos: {filas}")
print(f"Total de venta atribuida: {tottal_venta:,.2f}")
# Resultado final
df_atribucion.head()

Número de clientes que hicieron clic hasta 2026-09-30: 774
Clientes Efectivos: 24
Total de venta atribuida: 6,740,045.67


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,campania,fecha_envio,clics,dias_post_apertura
0,104116,C45692165,CLAUDIA HELENA GUAPI,claudiaguapi78@gmail.com,3008102563,2026-09-30,CHATCENTER WEB,173311.88,ULT DIA ENVIO GRATIS SEPT,2026-09-30,NaT,0
1,104118,C1018419250,ZARETH GALVAN,zcgalvan@gmail.com,3243670638,2026-09-30,Shopify,190958.17,ULT DIA ENVIO GRATIS SEPT,2026-09-30,NaT,0
2,104122,C45692165,CLAUDIA HELENA GUAPI,claudiaguapi78@gmail.com,3008102563,2026-09-30,CHATCENTER WEB,57980.70,ULT DIA ENVIO GRATIS SEPT,2026-09-30,NaT,0
3,104126,C19189694,JOSÉ LEONARDO SANSÓN G,leonardosanzon@gmail.com,3006157292,2026-09-30,CHATCENTER WEB,161967.83,ULT DIA ENVIO GRATIS SEPT,2026-09-30,NaT,0
4,104127,C66711820,LUZ PIEDAD CARDONA B.,luzpica@yahoo.com,3104349975,2026-09-30,CHATCENTER WEB,298096.42,ULT DIA ENVIO GRATIS SEPT,2026-09-30,NaT,0


In [187]:
df_atribucion.to_excel('Ventas.xlsx', index=False)